# PPG-derived respiration during HRV quiet periods

Compare three beat-derived respiratory modulations and a **direct raw-PPG RIIV alternative**. This notebook
uses the existing Polar channel-0 loader, movement detector, pulse detector,
interval artifact flags, and HRV quiet/window helpers. The whole recording is
analyzed with **sleep unverified**, as in the preceding Polar analysis.

The conventional terminology differs from the initial request:

| Method | Modulation | Signal used here |
|---|---|---|
| **RIIV** | Respiratory-induced **intensity** variation: baseline wander | Mean of a peak and its preceding foot on baseline-preserving PPG |
| **RIAV** | Respiratory-induced **amplitude** variation | Peak minus preceding foot on baseline-preserving PPG |
| **RIFV** | Respiratory-induced **frequency** variation: pulse-rate variability / RSA proxy | Onset-to-onset pulse period (PP), in seconds |
| **RIIV_direct** | Direct baseline/intensity extraction | Raw PPG low-pass filtered below each window's cardiac frequency |

RIIV is also extracted from peak intensity in some publications. We use the
peak/foot midline variant to reduce direct mixing with amplitude modulation.
No physical component is perfectly isolated: changing pulse shape can affect
both baseline and amplitude estimates. RIFV uses PP rather than its reciprocal
HR, retaining the convention of the existing pipeline; their fluctuation signs differ.

References:

- [Charlton et al.: Breathing Rate Estimation from the ECG and PPG, Table II](https://kclpure.kcl.ac.uk/portal/files/100427565/Charlton_et_al._Breathing_Rate_Estimation_from_the_Electrocardiogram_and_Photoplethysmogram_A_Review_2_.pdf)
  describes peak/foot means, differences, and pulse timing as extraction features.
- [Pimentel et al.: Toward a Robust Estimation of Respiratory Rate From Pulse Oximeters](https://pmc.ncbi.nlm.nih.gov/articles/PMC6051482/)
  defines RIIV/RIAV/RIFV and emphasizes signal quality. This notebook compares
  separate estimates; it does not reproduce that paper's AR fusion algorithm.

## Configuration

Your selected quiet policy matches **HRV**: ignore bursts shorter than 2 s,
exclude 1 s after retained bursts, and use 1–5-minute windows at 60 s steps.
The shared HRV helper adds a final end-anchored window when needed. Quiet
acquisition edges are included, matching HRV; set `include_recording_edges=False`
to require periods strictly between qualifying bursts. Ignored short movements
remain inside these periods by design and may still affect the PPG.

The Polar detector retains the earlier Polar defaults (52 Hz regularization,
20 mg envelope threshold). This matches the **selection policy** of HRV, not a
claim that a Garmin wrist threshold is validated for the Polar wear site.

All analysis settings are explicit below. Default respiratory search range:
**0.10–0.50 Hz / 6–30 breaths/min**. It is configurable and does not establish
that all breathing rates, or all physiologic conditions, lie in this range.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROCESSING = Path.cwd() if (Path.cwd() / "notebook_paths.py").exists() else Path.cwd() / "offline_processing"
PROCESSING = PROCESSING.resolve()
if not (PROCESSING / "notebook_paths.py").exists():
    raise FileNotFoundError("Run from the repository root or offline_processing")
if str(PROCESSING) not in sys.path:
    sys.path.insert(0, str(PROCESSING))

from notebook_paths import input_path
from polar_interpretability import load_polar_recording
from polar_ccf import sensor_clock_index
from analysis_windows import intersect_intervals, validate_intervals
from detect_acc_bursts import detect_bursts_in_intervals
from ppg_respiration import (
    METHODS, LABELS, RespirationConfig, analyze_respiration,
    respiration_availability, respiration_quiet_periods,
)
from direct_riiv import DIRECT_METHOD, DirectRIIVConfig, add_direct_riiv
from respiration_plots import (
    plot_comparison, plot_direct_filter, plot_example, plot_recording, plot_waveforms, representative_window,
)

CONFIG = RespirationConfig(
    min_burst_duration_s=2, post_burst_guard_s=1,
    min_window_s=60, max_window_s=300, step_s=60,
)
DIRECT_CONFIG = DirectRIIVConfig(
    cutoff_hr_fraction=.70, hr_quantile=.10, order=8, context_s=30,
    max_cutoff_hz=1.0, peak_cutoff_fraction=.85,
)
ACC_FS = 52.0
ACC_THRESHOLD_G = .020
MAX_ACC_GAP_S = .25
SENSOR_CLOCK_ZONE = "UTC"  # Bookkeeping label for sensor time; civil alignment is unverified.
REPRESENTATIVE_WINDOW_ID = None  # Automatic selection by QC coverage, never by agreement/rate.
OUTPUT_ROOT = PROCESSING / "outputs" / "respiration"
plt.style.use("seaborn-v0_8-whitegrid")
display(pd.Series(asdict(CONFIG), name="Respiration configuration"))
display(pd.Series(asdict(DIRECT_CONFIG), name="Direct RIIV filter configuration"))

## Load raw PPG and existing accelerometer processing

Use `verity_sense_path` in the ignored `paths.local.json`, or
`OVERNIGHT_VERITY_SENSE_PATH`, exactly as for the other Polar notebooks. Install
`offline_processing/requirements-notebooks.txt` in the kernel environment.
No new package or respiration model is needed.

Both streams keep the common Polar clock (nanoseconds from its 2000 epoch).
There is no Garmin clock shift or alignment by maximizing a physiological
correlation. The loader rejects missing/unsorted/duplicate PPG timestamps and
large recording gaps. Acceleration gaps above the configured limit fail
explicitly. A smaller raw PPG gap inside a quiet period is separately screened
when that period is processed; rejected periods stay in the diagnostics.

In [ ]:
raw_ppg, raw_acc = load_polar_recording(input_path("verity_sense_path"))
raw_ppg, raw_acc = raw_ppg.copy(), raw_acc.copy()
raw_ppg.index = sensor_clock_index(raw_ppg.index, SENSOR_CLOCK_ZONE)
raw_acc.index = sensor_clock_index(raw_acc.index, SENSOR_CLOCK_ZONE)
alignment_rows = []
for label, series in (("PPG", raw_ppg), ("ACC", raw_acc)):
    if not np.isfinite(series.to_numpy()).all():
        raise ValueError(f"Nonfinite {label} samples")
    gaps = np.diff(series.index.asi8) / 1e9
    alignment_rows.append({"stream": label, "start": series.index[0], "last_sample": series.index[-1],
                           "samples": len(series), "observed_median_hz": 1 / np.median(gaps),
                           "max_gap_s": gaps.max()})
alignment = pd.DataFrame(alignment_rows)
common_start, common_end = max(raw_ppg.index[0], raw_acc.index[0]), min(raw_ppg.index[-1], raw_acc.index[-1])
if common_end <= common_start:
    raise ValueError("No common PPG/ACC acquisition support")
acc_table = raw_acc.rename(columns={"x": "accel_x", "y": "accel_y", "z": "accel_z"})
acc_table = acc_table.rename_axis("sample_time").reset_index()
acquisition = validate_intervals(pd.DataFrame({"start": [raw_acc.index[0]], "end": [raw_acc.index[-1]]}))
detection = detect_bursts_in_intervals(
    acc_table, acquisition, sampling_rate=ACC_FS, input_unit="mg", max_gap_s=MAX_ACC_GAP_S,
    alfa=ACC_THRESHOLD_G, merge_gap_s=5,
)
coverage = intersect_intervals(detection["intervals"], pd.DataFrame({"start": [common_start], "end": [common_end]}))
quiet, quiet_report = respiration_quiet_periods(detection["bursts"], coverage, CONFIG)
display(alignment)
display(pd.Series(quiet_report, name="Movement-only availability"))
print("Whole recording; sleep UNVERIFIED. Quiet periods follow the selected HRV policy.")
display(quiet.head(10))

## Extract and quality-screen the three respiratory signals

Pulse detection reuses `polar_ccf.extract_beats` and the shared Aboy++-style
fiducial implementation. It runs **separately inside each quiet period**, using
the existing 0.5–8 Hz pulse-detection waveform and interval artifact flags.

That high-passed waveform is **not used to measure baseline wander or raw pulse
amplitude**. A parallel copy of the raw optical samples is regularized at 55 Hz,
sign-reversed according to the existing Polar convention, and low-pass filtered
at 8 Hz (order 3, SOS forward/reverse). It preserves DC and respiratory baseline.
RIIV and RIAV are measured there at the same paired peak and foot times.

RIFV is the measured onset-to-onset PP at its temporal midpoint; RIIV/RIAV retain
the peak time. Each modulation is independently linearly interpolated onto a
common **4 Hz** grid without extrapolation. No invented first PP, corrected
peak positions, averaging of separate periods, or synchronized shifting is used.

Common support makes the three comparisons use the same detected pulses.
Invalid or flagged pulse intervals are masked. A gap >3 s in either amplitude
or PP support creates a new run; neither filtering nor windows cross it. Short
masked gaps are interpolated, their fraction is reported, and windows with >5%
flagged support are rejected by default. The inherited classifier can flag true
RSA, especially when small PP variations are quantized at 55 Hz; inspect the
exclusion rates rather than treating artifact labels as physiological truth.

The three 4 Hz signals are filtered independently per continuous run with an
order-3 Butterworth bandpass at 0.10–0.50 Hz, using SOS forward/reverse filtering
and up to 30 s odd-reflection padding. Double passing squares the magnitude
response and avoids phase delay. We trim 10 s at each respiratory filter edge,
in addition to the pulse detector's 5 s waveform edge guard. This loses usable
duration, which the period/window tables report. Filtering cannot eliminate all
finite-run edge effects. Short periods are never padded into full windows.

## Direct RIIV: raw PPG low-pass filtered below window HR

This implements the requested experiment **alongside the existing feature
midline RIIV**. Each candidate window gets its own filter:

1. Reuse valid within-window PP intervals to estimate HR (`60 / PP`). Only the
   cutoff depends on beats; the RIIV waveform is computed from the raw optical
   samples, without peak/foot averaging or beat-rate decimation.
2. Use the **10th percentile of HR** as a conservative cardiac reference,
   `f_card = quantile(HR_bpm, 0.10) / 60` Hz. Set
   **`f_cut = min(0.70 × f_card, 1.0 Hz)`**. For example, a 45 bpm reference gives
   a 0.525 Hz cutoff. Both the reference quantile and multiplier are configurable.
3. Regularize raw PPG at 55 Hz, remove its context mean, and apply an **8th-order
   low-pass Butterworth using SOS forward/reverse filtering**. This is zero phase
   and effective order 16, with −6 dB response at the cutoff. Only after this
   cardiac suppression is the signal downsampled to 4 Hz.
4. Use up to 30 s of real context on each side **inside the same quiet period**,
   plus odd-reflection padding. Never read across a quiet boundary or raw gap.
   Overlapping windows can have different cutoffs, so their waveforms are saved
   separately by window ID rather than merged into one continuous night signal.
5. Estimate the rate from the low-pass output's detrended Hann spectrum, keeping
   the same 0.10–0.50 Hz search band. For display, also apply the existing
   respiratory bandpass. The adaptive low-pass can attenuate fast respiration;
   flag a peak at/above **85% of its cutoff** or theoretical cardiac attenuation
   below 30 dB. These are explicit experimental quality settings.

This is a custom HR-guided filter experiment, not an exact implementation of a
published algorithm. [Charlton et al. (2016), Table 1](https://www.robots.ox.ac.uk/~davidc/pubs/physmeas2016.pdf)
distinguishes direct filter-based baseline extraction from beat-feature methods.
The multiplier and quality thresholds here are engineering choices, not values
validated by that paper.

The direct signal is **not subject to a beat-sampling Nyquist rejection** because
it comes from native-rate PPG. Pulse quality still determines the reliability
of its HR estimate. For a matched comparison, this experiment uses exactly the
existing candidate windows; it does not yet measure additional raw-only coverage
where pulse detection failed. A subcardiac fluctuation can still be vasomotion,
contact/gain drift, or another artifact rather than respiration.

In [ ]:
result = analyze_respiration(raw_ppg, quiet, CONFIG)
result = add_direct_riiv(result, raw_ppg, DIRECT_CONFIG)
availability = respiration_availability(result)
display(availability)
display(result["periods"].groupby("status").size().rename("quiet_periods"))
if not result["windows"].empty:
    display(result["windows"].groupby(["method", "included", "exclusion_reason"]).size().rename("method_windows"))
else:
    warnings.warn("No complete 1–5-minute windows after pulse QC and edge trimming. Review diagnostics; geometry has not been relaxed.")
print(f"Movement-only windows: {quiet_report['geometric_windows']}; signal-supported windows: {result['windows'].window_id.nunique()}.")
display(result["windows"].sort_values(["window_id", "method"]).head(12))
direct_rows = result["windows"].loc[result["windows"].method == DIRECT_METHOD]
display(direct_rows[["hr_median_bpm", "hr_filter_reference_bpm", "lowpass_cutoff_hz",
                     "cardiac_attenuation_db_theoretical", "candidate_rate_bpm", "included"]].describe(include="all"))

## Breathing-rate estimation and quality measures

For comparison, each window's **unfiltered beat-derived modulation** is demeaned
and linearly detrended. Its full-window Hann periodogram identifies the strongest
frequency in the configured search range. No zero padding is used: resolution
is `60 / window_duration_s` breaths/min. The respiratory bandpass above is used
to display waveforms; it does not bias the spectral peak selection toward the
center of the filter passband.

Both the candidate peak and the quality-screened rate are retained. The latter
is missing when pulse support fails QC, variance is absent, the peak is at a
search boundary, or the fraction of in-band power within ±0.025 Hz of the peak
is <0.20. The concentration threshold is an explicit exploratory screen, not a
calibrated confidence score. Multi-peak spectra, harmonics and non-respiratory
vasomotor fluctuations can still mislead the estimates.

**For the three beat-derived methods**, interpolation at 4 Hz does not restore frequencies lost by beat sampling. Peaks
at or above 90% of an **approximate beat-sampling Nyquist** (`0.5 / median_PP`)
are flagged. This conservative diagnostic does not prove that lower-frequency
peaks are free of aliasing in irregular beat sampling. `RIIV_direct` instead uses native-rate low-pass filtering before downsampling and its cutoff/attenuation checks.

## Whole-recording overview and estimated rates

All detected movements remain visible, including short bursts ignored by the
selected HRV quiet policy. The second panel distinguishes geometric quiet
periods from continuous signal support remaining after pulse QC and edge
trimming. Rates are shown as points; no line joins separate quiet periods.

In [ ]:
fig_overview = plot_recording(detection, result)
plt.show()

## Representative window: raw PPG, modulations, spectra

Automatic selection favors the greatest number of methods passing QC (including the direct variant), then the
lowest flagged fraction and earliest time. It does **not** select on agreement,
estimated rate, or a desired phase relation. Set `REPRESENTATIVE_WINDOW_ID` to
inspect any other candidate. Z-scores are for display only; saved features keep
their optical units or seconds. Sign and phase do not identify inspiration or
expiration without a reference signal.

In [ ]:
representative = representative_window(result, REPRESENTATIVE_WINDOW_ID)
fig_example = plot_example(raw_ppg, result, representative)
plt.show()
if representative is not None:
    display(representative[["window_id", "quiet_period_id", "method", "duration_s",
                            "candidate_rate_bpm", "rate_bpm", "peak_power_fraction", "included", "exclusion_reason"]])

## Respiration waveforms in a quiet period

The three beat-derived modulations below each have their own scale. Direct RIIV is shown per window in the next section because its filter changes between windows. Waveforms remain separate across gaps; no
breaths are inferred over missing support. These curves represent PPG-derived
respiratory modulation, not airflow, respiratory volume, or a direct respiratory
belt measurement.

In [ ]:
selected_quiet_id = None if representative is None else int(representative.quiet_period_id.iloc[0])
fig_waveforms = plot_waveforms(result, selected_quiet_id, max_minutes=10)
plt.show()

## Inspect the HR-adaptive direct RIIV filter

Compare raw PPG with the low-pass baseline, then compare direct RIIV with the
feature-midline RIIV on the same window. Native-rate spectra show suppression
near the cardiac reference and the position of the cutoff relative to the
respiratory band. The reported attenuation is the filter's theoretical gain at
that reference frequency; it does not guarantee removal of a changing pulse
rate or artifacts.

In [ ]:
fig_direct_filter = plot_direct_filter(raw_ppg, result, representative)
plt.show()
if representative is not None:
    display(representative.loc[representative.method == DIRECT_METHOD,
                               ["window_id", "hr_median_bpm", "hr_filter_reference_bpm", "lowpass_cutoff_hz",
                                "cardiac_attenuation_db_theoretical", "respiratory_flat_band_max_hz",
                                "candidate_rate_bpm", "included", "exclusion_reason"]])

## Compare matched windows and quiet-period summaries

Coverage varies by method, so paired plots use only windows passing **all four**
methods, including the new direct RIIV alternative. The scatter plots compare direct RIIV with each feature method. The descriptive pairwise table first takes a median within each quiet
period, then compares those period medians. This avoids counting every
overlapping window as an independent observation. Per-method summaries using all
accepted windows are also saved and explicitly have different coverage.

No accuracy ranking is possible without reference respiration. Agreement among
the methods can reflect shared optical artifacts. This single participant's
quiet periods are not proven independent, and no independent-window significance
tests or confidence intervals are reported.

In [ ]:
fig_comparison = plot_comparison(result)
plt.show()
window_meta = result["windows"].drop_duplicates("window_id").set_index("window_id")[["quiet_period_id", "start", "end", "duration_s"]]
comparison = result["comparison"].join(window_meta)
matched = comparison.loc[comparison.all_methods_included]
comparison_methods = result.get("methods", METHODS)
matched_period_rates = matched.groupby("quiet_period_id")[list(comparison_methods)].median()
pair_rows = []
for a, b in combinations(comparison_methods, 2):
    delta = matched_period_rates[b] - matched_period_rates[a]
    pair_rows.append({"pair": f"{b} minus {a}", "matched_windows": len(matched),
                      "distinct_quiet_periods": len(matched_period_rates),
                      "median_difference_bpm": delta.median(), "median_absolute_difference_bpm": delta.abs().median()})
pair_summary = pd.DataFrame(pair_rows)
print(f"Windows passing all {len(comparison_methods)} methods: {len(matched)}; distinct contributing quiet periods: {len(matched_period_rates)}.")
print(f"Original three methods passing: {int(comparison.all_feature_methods_included.sum())} windows.")
if len(matched_period_rates) < 2:
    warnings.warn("Fewer than two quiet periods with common four-method results; comparison is limited.")
display(pair_summary)
display(matched_period_rates)

## Export reproducible local results

All data-dependent output stays in ignored `offline_processing/outputs/respiration/`.
Window-level CSV includes timestamps, quiet/run IDs, method, duration, candidate
and screened rate, exclusions, pulse QC, spectral concentration, frequency
resolution, and beat-Nyquist diagnostic. Separate CSVs retain unfiltered and
respiratory-filtered 4 Hz waveforms, per-beat optical features and PP, spectra,
quiet-period diagnostics, and matched comparisons. Source hashes and configuration
are saved with package versions. Direct-filter waveforms are stored in a separate per-window CSV with repeated times allowed across different window IDs. Its native PPG spectra and HR/cutoff diagnostics are also saved. The new `_direct` output directory preserves the earlier three-method results.

In [ ]:
run_dir = OUTPUT_ROOT / f"{raw_ppg.index[0]:%Y%m%dT%H%M%S}_hrv_quiet_direct"
run_dir.mkdir(parents=True, exist_ok=True)
tables = {
    "window_rates": result["windows"], "respiration_waveforms": result["signals"],
    "direct_riiv_waveforms": result["direct_signals"], "direct_native_ppg_spectra": result["direct_native_spectra"],
    "beat_features": result["features"], "window_spectra": result["spectra"],
    "quiet_periods": result["periods"].reset_index(), "movement_bursts": detection["bursts"],
    "period_rates": result["period_rates"], "availability": availability,
    "stream_alignment": alignment, "paired_comparison": comparison.reset_index(),
    "matched_period_rates": matched_period_rates.reset_index(), "pair_summary": pair_summary,
}
for name, table in tables.items():
    table.to_csv(run_dir / f"{name}.csv", index=False)
for name, figure in {"01_recording": fig_overview, "02_example": fig_example,
                      "03_waveforms": fig_waveforms, "04_comparison": fig_comparison, "05_direct_filter": fig_direct_filter}.items():
    figure.savefig(run_dir / f"{name}.png", dpi=160, bbox_inches="tight")
try:
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROCESSING, text=True).strip()
except (OSError, subprocess.CalledProcessError):
    revision = None
metadata = {
    "config": asdict(CONFIG), "direct_riiv_config": asdict(DIRECT_CONFIG), "quiet_report": quiet_report,
    "sleep_status": "whole recording; sleep unverified", "sensor_clock_zone": SENSOR_CLOCK_ZONE,
    "acc_fs": ACC_FS, "acc_threshold_g": ACC_THRESHOLD_G, "max_acc_gap_s": MAX_ACC_GAP_S,
    "methods": {"RIIV": "raw peak/foot midline", "RIAV": "raw peak minus foot", "RIFV": "PP at interval midpoint", "RIIV_direct": "native raw PPG, window-specific HR-adaptive low-pass"},
    "rate_estimator": "feature modulations or adaptive-low-pass raw PPG; detrended full-window Hann periodogram; no zero padding",
    "git_revision": revision, "python": platform.python_version(),
    "source_sha256": {name: hashlib.sha256((PROCESSING / name).read_bytes()).hexdigest()
                       for name in ["respiration.ipynb", "ppg_respiration.py", "direct_riiv.py", "respiration_plots.py", "polar_ccf.py",
                                    "ppg_fiducials.py", "gp_hrv.py", "gp_hrv_artifacts.py", "detect_acc_bursts.py"]},
    "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "matplotlib", "neurokit2"]},
    "reference_respiration": "none; agreement is not accuracy", "overlapping_windows_independent": False,
}
(run_dir / "analysis_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print(f"Saved waveforms, rate estimates, diagnostics, spectra, and five figures to {run_dir}")

## What these results support

This notebook supports checking whether the three PPG modulations exhibit
consistent respiratory-band rhythms during the chosen quiet periods, and how
their waveform shapes, spectral peaks, quality, and coverage differ. Baseline
drift, contact pressure, pulse morphology, vasomotion and device gain control
can alter RIIV/RIAV; PP quantization, pulse-arrival variation and artifact
classification can alter RIFV. Absence of a clear RIFV rhythm does not establish
absence of breathing. No apnea, respiratory volume, sleep-stage, or causal
conclusion follows without appropriate reference measurements.